In [1]:
import os
import sys
from pathlib import Path
import torch
import pandas as pd

# --- Phase 1: Environment and Path Preflight ---

def phase_1_preflight():
    print("=== Phase 1: Environment and Path Preflight ===")
    
    # 1. Detect project root
    cwd = Path.cwd()
    project_root = None
    for parent in [cwd] + list(cwd.parents):
        if (parent / "pyproject.toml").exists():
            project_root = parent
            break
    
    if not project_root:
        print("FAIL: Project root not found (missing pyproject.toml)")
        return False

    # 2. Define Required Files (Canonical Paths)
    required_files = {
        "model_input": "data/processed/model_input/model_input_node_hourly_features.parquet",
        "graph_pyg": "data/processed/graph/topology_graph_pyg_inference.pt",
        "gapfill": "data/processed/gapfill/station_timeseries_observed_imputed.parquet",
        "station_map": "data/processed/graph/station_to_topology_node_map.parquet",
        "node_id_map": "data/processed/graph/topology_nodeid_to_index_map.parquet"
    }
    
    missing = []
    for keys, rel_path in required_files.items():
        full_path = project_root / rel_path
        if not full_path.exists():
            missing.append(str(rel_path))
            
    # 3. System Info
    print(f"Python: {sys.version.split()[0]}")
    print(f"Torch: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Project Root: {project_root}")
    print(f"CWD: {cwd}")

    if missing:
        print("\nFAIL: Missing Required Files:")
        for m in missing:
            print(f"  - {m}")
        return False

    print("\nPASS: All preflight checks complete.")
    return True

# Execute Phase 1
p1_success = phase_1_preflight()

=== Phase 1: Environment and Path Preflight ===
Python: 3.9.7
Torch: 2.8.0+cu128
CUDA Available: True
GPU: Tesla V100-PCIE-32GB
Project Root: /home/jupyter-1nt23cb058/Capstone
CWD: /home/jupyter-1nt23cb058/Capstone/notebooks

PASS: All preflight checks complete.


In [2]:
# --- Phase 2: Canonical Loading ---

def phase_2_loading():
    print("=== Phase 2: Canonical Loading ===")
    root = Path("/home/jupyter-1nt23cb058/Capstone")
    
    try:
        # 1. Load model_input
        mi_path = root / "data/processed/model_input/model_input_node_hourly_features.parquet"
        df_model_input = pd.read_parquet(mi_path)
        
        # 2. Load station year files (Excluding 2023)
        station_dir = root / "data/processed/stations"
        station_years = ["2022.parquet", "2024.parquet", "2025.parquet", "2026_partial.parquet"]
        df_stations = pd.concat([pd.read_parquet(station_dir / y) for y in station_years])
        
        # 3. Load gapfill reference
        gf_path = root / "data/processed/gapfill/station_timeseries_observed_imputed.parquet"
        df_gapfill = pd.read_parquet(gf_path)
        
        # 4. Load graph with safe globals / weights_only handling for Torch 2.6+
        # Using weights_only=False because PyG objects contain custom classes
        graph_path = root / "data/processed/graph/topology_graph_pyg_inference.pt"
        graph = torch.load(graph_path, weights_only=False)
        
        # Shapes table
        shapes = {
            "model_input": df_model_input.shape,
            "stations_concat": df_stations.shape,
            "gapfill": df_gapfill.shape,
            "graph_nodes": graph.num_nodes if hasattr(graph, 'num_nodes') else "N/A",
            "graph_edges": graph.edge_index.shape[1] if hasattr(graph, 'edge_index') else "N/A"
        }
        
        print(pd.Series(shapes).to_string())
        
        # Checks
        c1 = not df_model_input.empty
        c2 = hasattr(graph, 'edge_index') and graph.edge_index.shape[0] == 2
        c3 = not df_stations.empty
        c4 = not df_gapfill.empty
        
        if all([c1, c2, c3, c4]):
            print("\nPASS: All canonical artifacts loaded successfully.")
            return df_model_input, df_stations, df_gapfill, graph
        else:
            print(f"\nFAIL: Content check failed. MI: {c1}, Graph: {c2}, Stations: {c3}, Gapfill: {c4}")
            return None
            
    except Exception as e:
        print(f"\nFAIL: Error during loading: {str(e)}")
        return None

# Execute Phase 2
load_results = phase_2_loading()
if load_results:
    df_model_input, df_stations, df_gapfill, graph = load_results

=== Phase 2: Canonical Loading ===
model_input        (1932700, 21)
stations_concat    (1073807, 14)
gapfill            (1073807, 14)
graph_nodes               154902
graph_edges               424550

PASS: All canonical artifacts loaded successfully.


In [3]:
# --- Phase 3: Schema and dtype audit (Updated) ---

def phase_3_audit(df_mi, df_st, df_gf):
    print("=== Phase 3: Schema and dtype audit ===")
    
    # Diagnostic: Print columns to resolve the mapping
    print(f"Model Input Columns: {list(df_mi.columns[:5])}...")
    print(f"Stations Columns: {list(df_st.columns[:5])}...")
    print(f"Gapfill Columns: {list(df_gf.columns[:5])}...")

    try:
        # 1. Column Resolution Map
        # We need to find which column represents the timestamp in stations/gapfill
        st_ts_col = 'timestamp' if 'timestamp' in df_st.columns else ('utc' if 'utc' in df_st.columns else None)
        gf_ts_col = 'timestamp' if 'timestamp' in df_gf.columns else ('utc' if 'utc' in df_gf.columns else None)
        
        if not st_ts_col or not gf_ts_col:
            # If still not found, check for any column with 'time' in it
            st_ts_col = [c for c in df_st.columns if 'time' in c.lower()][0]
            gf_ts_col = [c for c in df_gf.columns if 'time' in c.lower()][0]

        print(f"Resolved Timestamp Columns -> ST: {st_ts_col}, GF: {gf_ts_col}")

        # 2. Timestamp Coercion (UTC-Naive)
        def clean_ts(df, col_name, target_name):
            # Convert to datetime, then to naive UTC
            df['timestamp_clean'] = pd.to_datetime(df[col_name], utc=True).dt.tz_localize(None)
            # Standardize column name to 'timestamp' for the audit
            df.drop(columns=[col_name], inplace=False) 
            null_count = df['timestamp_clean'].isnull().sum()
            return null_count

        mi_nulls = clean_ts(df_mi, 'timestamp', "model_input")
        st_nulls = clean_ts(df_st, st_ts_col, "stations")
        gf_nulls = clean_ts(df_gf, gf_ts_col, "gapfill")
        
        # Rename for internal consistency in later phases
        df_mi['timestamp'] = df_mi['timestamp_clean']
        df_st['timestamp'] = df_st['timestamp_clean']
        df_gf['timestamp'] = df_gf['timestamp_clean']

        # 3. Numeric Enforcement
        df_mi['node_id'] = df_mi['node_id'].astype(str)
        df_st['station_id'] = df_st['station_id'].astype(str)
        df_gf['station_id'] = df_gf['station_id'].astype(str)

        # 4. Duplicate Check
        mi_dupes = df_mi.duplicated(subset=['timestamp', 'node_id']).sum()
        st_dupes = df_st.duplicated(subset=['timestamp', 'station_id']).sum()
        gf_dupes = df_gf.duplicated(subset=['timestamp', 'station_id']).sum()

        # Reporting
        report = {
            "MI Null Timestamps": mi_nulls,
            "ST Null Timestamps": st_nulls,
            "GF Null Timestamps": gf_nulls,
            "MI Duplicate Keys": mi_dupes,
            "ST Duplicate Keys": st_dupes,
            "GF Duplicate Keys": gf_dupes
        }
        print(pd.Series(report).to_string())

        if (mi_nulls + st_nulls + gf_nulls) == 0:
            print("\nPASS: Schema and dtypes are consistent.")
            return True
        else:
            print("\nFAIL: Null timestamps detected.")
            return False

    except Exception as e:
        print(f"\nFAIL: Error during audit: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# Execute Phase 3
p3_success = phase_3_audit(df_model_input, df_stations, df_gapfill)

=== Phase 3: Schema and dtype audit ===
Model Input Columns: ['node_id', 'timestamp', 'station_id', 'station_name', 'station_co']...
Stations Columns: ['time', 'station_id', 'value', 'parameter', 'observed_value']...
Gapfill Columns: ['time', 'station_id', 'value', 'parameter', 'observed_value']...
Resolved Timestamp Columns -> ST: time, GF: time
MI Null Timestamps         0
ST Null Timestamps         0
GF Null Timestamps         0
MI Duplicate Keys          0
ST Duplicate Keys     725985
GF Duplicate Keys     725985

PASS: Schema and dtypes are consistent.


In [4]:
# --- Phase 4: Coverage and chronology audit ---

def phase_4_coverage(df_mi, df_st, df_gf):
    print("=== Phase 4: Coverage and Chronology Audit ===")
    
    try:
        # 1. Validate 2023 Absence in stations
        st_years = df_st['timestamp'].dt.year.unique()
        has_2023 = 2023 in st_years
        
        # 2. Year Range Extraction
        mi_min, mi_max = df_mi['timestamp'].min(), df_mi['timestamp'].max()
        st_min, st_max = df_st['timestamp'].min(), df_st['timestamp'].max()
        gf_min, gf_max = df_gf['timestamp'].min(), df_gf['timestamp'].max()
        
        # 3. Check Overlap
        st_keys = set(df_st['station_id'].unique())
        gf_keys = set(df_gf['station_id'].unique())
        key_overlap = len(st_keys.intersection(gf_keys))

        # Reporting
        print(f"Model Input Range: {mi_min} to {mi_max}")
        print(f"Stations Range:    {st_min} to {st_max}")
        print(f"Gapfill Range:     {gf_min} to {gf_max}")
        print(f"Station Years Found: {sorted(st_years)}")
        print(f"Station-Gapfill ID Overlap: {key_overlap}")

        # Checks
        c1 = not has_2023
        c2 = key_overlap > 0
        c3 = mi_min >= pd.Timestamp('2022-01-01')

        if c1 and c2 and c3:
            print("\nPASS: Coverage and chronology constraints met.")
            return True
        else:
            errors = []
            if has_2023: errors.append("2023 detected in stations")
            if key_overlap == 0: errors.append("No ID overlap between stations and gapfill")
            if not c3: errors.append(f"Model input starts before 2022: {mi_min}")
            print(f"\nFAIL: {', '.join(errors)}")
            return False

    except Exception as e:
        print(f"\nFAIL: Error during coverage audit: {str(e)}")
        return False

# Execute Phase 4
p4_success = phase_4_coverage(df_model_input, df_stations, df_gapfill)
# --- Fix for Phase 4: Temporal Truncation ---

print("Applying fix: Truncating model_input to >= 2022-01-01")
original_count = len(df_model_input)
df_model_input = df_model_input[df_model_input['timestamp'] >= pd.Timestamp('2022-01-01')].copy()
new_count = len(df_model_input)

print(f"Rows removed: {original_count - new_count}")
print(f"New Model Input Range: {df_model_input['timestamp'].min()} to {df_model_input['timestamp'].max()}")

# Re-run Phase 4 check logic
p4_success = (
    2023 not in df_stations['timestamp'].dt.year.unique() and
    df_model_input['timestamp'].min() >= pd.Timestamp('2022-01-01')
)

if p4_success:
    print("\nPASS: Coverage and chronology constraints met after truncation.")

=== Phase 4: Coverage and Chronology Audit ===
Model Input Range: 2016-03-21 18:00:00 to 2026-03-30 03:00:00
Stations Range:    2022-01-01 00:00:00 to 2026-03-31 05:00:00
Gapfill Range:     2022-01-01 00:00:00 to 2026-03-31 05:00:00
Station Years Found: [np.int32(2022), np.int32(2024), np.int32(2025), np.int32(2026)]
Station-Gapfill ID Overlap: 17

FAIL: Model input starts before 2022: 2016-03-21 18:00:00
Applying fix: Truncating model_input to >= 2022-01-01
Rows removed: 1114740
New Model Input Range: 2022-01-01 00:00:00 to 2026-03-30 03:00:00

PASS: Coverage and chronology constraints met after truncation.


In [5]:
##=== Phase 5: Graph Integrity Audit ===
import torch
import pandas as pd
from pathlib import Path

# --- Exact Path Loading ---
root = Path("/home/jupyter-1nt23cb058/Capstone") 
graph_path = root / "data/processed/graph/topology_graph.pt"
graph_data = torch.load(graph_path)

# Check if it's a dict and extract the actual graph
if isinstance(graph_data, dict):
    print(f"Detected dictionary structure. Keys: {list(graph_data.keys())}")
    # Typically, the Data object is under a 'graph' or 'data' key, 
    # or the dict itself contains the tensors.
    if 'edge_index' in graph_data:
        # If the dict is the graph itself
        edge_index = graph_data['edge_index']
        num_nodes = graph_data.get('num_nodes', edge_index.max().item() + 1)
        # Reconstruct as a PyG Data object for the audit
        from torch_geometric.data import Data
        graph = Data(edge_index=edge_index, num_nodes=num_nodes)
    else:
        # Try to find a Data object inside the dict
        graph = next((v for v in graph_data.values() if hasattr(v, 'edge_index')), None)
else:
    graph = graph_data

def phase_5_graph_audit(pyg_graph):
    print("=== Phase 5: Graph Integrity Audit ===")
    if pyg_graph is None:
        print("FAIL: Could not extract a valid Graph object from the file.")
        return False
        
    try:
        edge_index = pyg_graph.edge_index
        num_nodes = pyg_graph.num_nodes
        
        is_correct_shape = (edge_index.shape[0] == 2)
        in_bounds = (edge_index.max().item() < num_nodes)
        
        stats = {
            "Num Nodes": num_nodes,
            "Num Edges": edge_index.shape[1],
            "Shape Correct": is_correct_shape,
            "In Bounds": in_bounds
        }
        print(pd.Series(stats).to_string())
        return is_correct_shape and in_bounds
    except Exception as e:
        print(f"FAIL: {e}")
        return False

p5_success = phase_5_graph_audit(graph)

Detected dictionary structure. Keys: ['x', 'edge_index', 'edge_attr', 'num_nodes', 'node_feature_schema', 'highway_categories', 'edge_attr_schema']
=== Phase 5: Graph Integrity Audit ===
Num Nodes        154902
Num Edges        424550
Shape Correct      True
In Bounds          True


In [6]:
## safety check

def safety_phase_unified_audit(root_path, pyg_graph):
    print("=== Safety Phase: Unified 3-Way Join Audit ===")
    root = Path(root_path)
    
    # 1. Load Mappings
    df_st_to_node = pd.read_parquet(root / "data/processed/graph/station_to_topology_node_map.parquet")
    df_node_to_idx = pd.read_parquet(root / "data/processed/graph/topology_nodeid_to_index_map.parquet")
    df_map = df_st_to_node.merge(df_node_to_idx, on='node_id', how='inner')
    df_map['location_id'] = df_map['location_id'].astype(str)

    years = [2022, 2024, 2025, 2026]
    master_chunks = []

    for year in years:
        try:
            suffix = "_partial" if year == 2026 else ""
            df_st = pd.read_parquet(root / f"data/processed/stations/{year}{suffix}.parquet")
            df_w  = pd.read_parquet(root / f"data/processed/weather/{year}{suffix}.parquet")
            df_aq = pd.read_parquet(root / f"data/processed/airquality/{year}{suffix}.parquet")
            
            # --- Robust Time Standardization ---
            def standardize_df(df, time_col_name='timestamp'):
                # If index is datetime, move it to a column
                if isinstance(df.index, pd.DatetimeIndex):
                    df = df.reset_index().rename(columns={df.index.name: time_col_name})
                # Ensure the column exists and is datetime
                if time_col_name in df.columns:
                    df[time_col_name] = pd.to_datetime(df[time_col_name]).dt.tz_localize(None)
                elif 'time' in df.columns:
                    df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
                return df

            df_st = standardize_df(df_st, 'time')
            df_st['location_id'] = df_st['station_id'].astype(str)
            df_w  = standardize_df(df_w, 'timestamp')
            df_aq = standardize_df(df_aq, 'timestamp')

            # --- Triple Join ---
            # A: Link stations to GNN Indices
            j1 = df_st.merge(df_map[['location_id', 'node_index']], on='location_id', how='inner')
            # B: Join Weather
            j2 = j1.merge(df_w, left_on='time', right_on='timestamp', how='inner')
            # C: Join Air Quality
            final_yr = j2.merge(df_aq, on='timestamp', how='inner')
            
            if not final_yr.empty:
                master_chunks.append(final_yr)
                print(f"Year {year}: {len(final_yr):,} rows successfully bridged.")
            else:
                print(f"Year {year}: Warning - Join resulted in 0 rows.")
                
        except Exception as e:
            print(f"Year {year} Failed: {str(e)}")

    if not master_chunks:
        print("CRITICAL: No data was successfully joined for any year.")
        return None

    full_df = pd.concat(master_chunks, ignore_index=True)
    
    # 2. Methodology: Abs Value + 250 Cap
    def digit_fix(v):
        if v >= 1000:
            s = str(int(v))
            return float(s[:3])
        return v
    full_df['y_clean'] = full_df['value'].apply(digit_fix).abs().clip(upper=250.0)
    
    # 3. Final Check
    max_idx = full_df['node_index'].max()
    is_safe = max_idx < pyg_graph.num_nodes
    
    print(f"\n--- AUDIT SUMMARY ---")
    print(f"Total Combined Samples: {len(full_df):,}")
    print(f"Max Node Index:       {max_idx} (Limit: {pyg_graph.num_nodes})")
    print(f"Readiness Status:     {'PASS' if is_safe else 'FAIL'}")
    
    return full_df

# Execute
full_report_df = safety_phase_unified_audit(root, graph)

=== Safety Phase: Unified 3-Way Join Audit ===
Year 2022: 269,462 rows successfully bridged.
Year 2024: 194,615 rows successfully bridged.
Year 2025: 487,379 rows successfully bridged.
Year 2026: 118,747 rows successfully bridged.

--- AUDIT SUMMARY ---
Total Combined Samples: 1,070,203
Max Node Index:       153451 (Limit: 154902)
Readiness Status:     PASS


In [8]:
# --- Phase 6: Global Feature Scaling ---
# Updated to handle non-numeric types and prevent the TypeError.
# This cell will only attempt to scale columns that contain actual numbers.

import joblib 
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import sys

def phase_6_global_scaling_safe(df_master, root_path):
    print("=== Phase 6: Global Feature Scaling ===")
    root = Path(root_path)
    
    # 1. Identify potential features
    exclude_cols = ['time', 'timestamp', 'station_id', 'location_id', 'node_index', 'value', 'y_clean']
    potential_cols = [c for c in df_master.columns if c not in exclude_cols]
    
    # 2. Strict Numeric Enforcement: Filter out strings/objects
    feature_cols = []
    for col in potential_cols:
        # Try to convert to numeric to test if it's actually math-ready
        if pd.api.types.is_numeric_dtype(df_master[col]):
            feature_cols.append(col)
        else:
            print(f"  ! Skipping non-numeric column: {col}")

    scaler_registry = {}
    
    # 3. Scaling Loop
    print(f"Normalizing {len(feature_cols)} validated numeric features...")
    for col in tqdm(feature_cols, desc="Scaling Progress", unit="feature"):
        try:
            # Ensure no NaNs are messing up the math
            raw_values = df_master[col].fillna(0).values
            mean = np.mean(raw_values)
            std = np.std(raw_values)
            
            # Create a new column with float32 to save memory
            df_master[f'{col}_scaled'] = ((raw_values - mean) / (std + 1e-8)).astype(np.float32)
            scaler_registry[col] = {'mean': mean, 'std': std}
        except Exception as e:
            print(f"  ! Failed to scale {col}: {e}")
        
    # 4. Target Scaling (PM2.5)
    print("Finalizing: Scaling target variable (y_clean)...")
    y_vals = df_master['y_clean'].fillna(0).values
    y_mean, y_std = np.mean(y_vals), np.std(y_vals)
    df_master['y_scaled'] = ((y_vals - y_mean) / (y_std + 1e-8)).astype(np.float32)
    scaler_registry['target'] = {'mean': y_mean, 'std': y_std}
    
    # 5. Save the Scaler Registry
    scaler_path = root / "data/processed/graph/global_scaler.pkl"
    scaler_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(scaler_registry, scaler_path)
    
    print(f"\nSUCCESS: Scaler saved to {scaler_path}")
    return df_master, feature_cols

# Run the Robust Version
df_scaled, scaled_feature_list = phase_6_global_scaling_safe(full_report_df, "/home/jupyter-1nt23cb058/Capstone")

# --- Save Point ---
# Run this immediately after success so you never have to run Phase 6 again!
root_path = Path("/home/jupyter-1nt23cb058/Capstone")
df_scaled.to_parquet(root_path / "data/processed/graph/master_scaled_checkpoint.parquet")
print("Master Checkpoint Saved Successfully.")

=== Phase 6: Global Feature Scaling ===
  ! Skipping non-numeric column: parameter
  ! Skipping non-numeric column: impute_method
  ! Skipping non-numeric column: impute_tier
  ! Skipping non-numeric column: impute_run_id
  ! Skipping non-numeric column: station_name
Normalizing 17 validated numeric features...


Scaling Progress:   0%|          | 0/17 [00:00<?, ?feature/s]

Finalizing: Scaling target variable (y_clean)...

SUCCESS: Scaler saved to /home/jupyter-1nt23cb058/Capstone/data/processed/graph/global_scaler.pkl
Master Checkpoint Saved Successfully.


In [9]:
# --- Phase 7: Temporal Windowing ---
# This cell transforms the flat 1.07M rows into 3D tensors for the GRU.
# Window size = 24 hours. We group by node_index to ensure sequences 
# don't accidentally "jump" from one station to another.

import torch
import numpy as np
from tqdm.notebook import tqdm
from pathlib import Path

def phase_7_temporal_windowing(df_scaled, feature_list, window_size=24):
    print(f"=== Phase 7: Temporal Windowing (Window Size: {window_size}h) ===")
    
    # 1. Use the '_scaled' versions of the features
    scaled_features = [f"{col}_scaled" for col in feature_list]
    
    # 2. Group by node and sort by time for continuous sequences
    # Using 'sort=False' on groupby can be faster if data is already sorted
    grouped = df_scaled.sort_values(['node_index', 'timestamp']).groupby('node_index')
    
    X_list, y_list, node_list = [], [], []
    
    print(f"Generating windows for {len(grouped)} unique sensor locations...")
    
    # 3. Sliding Window Loop
    # We iterate through each station's timeline
    for node_idx, group in tqdm(grouped, desc="Processing Node Timelines"):
        # Convert to numpy for maximum slicing speed
        feat_vals = group[scaled_features].values
        target_vals = group['y_scaled'].values
        
        # If a station has less than 24h of data, we skip it
        if len(group) <= window_size:
            continue
            
        # Create sliding windows: [t-23, ..., t0] -> [t+1]
        for i in range(len(group) - window_size):
            X_list.append(feat_vals[i : i + window_size])
            y_list.append(target_vals[i + window_size])
            node_list.append(node_idx)

    # 4. Final Tensor Conversion (Float32 is the V100 native language)
    print("Converting lists to Tensors (this may take a moment)...")
    X_train = torch.tensor(np.array(X_list), dtype=torch.float32)
    y_train = torch.tensor(np.array(y_list), dtype=torch.float32).view(-1, 1)
    node_indices = torch.tensor(np.array(node_list), dtype=torch.long)
    
    print(f"\nSUCCESS: Generated {X_train.shape[0]:,} sequences.")
    print(f"Tensor Shapes:\n  X: {X_train.shape} [Samples, Time, Features]\n  y: {y_train.shape}")
    
    return X_train, y_train, node_indices

# Execute Phase 7
X_train, y_train, node_ids = phase_7_temporal_windowing(df_scaled, scaled_feature_list)

=== Phase 7: Temporal Windowing (Window Size: 24h) ===
Generating windows for 17 unique sensor locations...


Processing Node Timelines:   0%|          | 0/17 [00:00<?, ?it/s]

Converting lists to Tensors (this may take a moment)...

SUCCESS: Generated 1,069,795 sequences.
Tensor Shapes:
  X: torch.Size([1069795, 24, 17]) [Samples, Time, Features]
  y: torch.Size([1069795, 1])


In [10]:
# --- Save Point: Training Tensors ---
# We save the final tensors so we can load them directly into the V100 notebook.

tensor_dir = Path("/home/jupyter-1nt23cb058/Capstone/data/processed/tensors")
tensor_dir.mkdir(parents=True, exist_ok=True)

torch.save(X_train, tensor_dir / "X_train.pt")
torch.save(y_train, tensor_dir / "y_train.pt")
torch.save(node_ids, tensor_dir / "node_ids.pt")

print(f"Tensors saved successfully to {tensor_dir}")
print("You are now safe to shut down this kernel and move to the new notebook.")

Tensors saved successfully to /home/jupyter-1nt23cb058/Capstone/data/processed/tensors
You are now safe to shut down this kernel and move to the new notebook.
